# 1. Data selection

**Component 3 – Temporal Product Quality & Emerging Issue Detection (PP1)**

Choose the smartphone and laptop subset that the rest of the component is built on, using the
Amazon Reviews 2023 dataset (McAuley Lab).

Selection rules (from the PP1 plan):

1. Smartphones come from *Cell Phones & Accessories*, laptops from *Electronics*.
2. A product is a **`parent_asin`** group, so colour/storage variants are not mixed up.
3. Keep products with **>= 500 reviews**, **>= 3 years** of review history and **>= 10 quarters with at least
   10 reviews** (the rolling-origin validation needs cutoffs such as Q1–Q8, Q1–Q9, Q1–Q10).
4. Selected sample: **up to 25 products per type** (at most 5 per brand), the proposal's 30–50 target.
5. **PP1 development subset:** 5 phones + 5 laptops from that sample (at most 2 per brand, one *sparse showcase*
   per type), flagged `pp1_subset` so the first iterations run fast.

Steps:

| Step | What happens | Output |
| --- | --- | --- |
| 1a | Stream each metadata file, keep only lines with enough ratings (resumable) | `data/raw/meta_<category>_prefilter.jsonl` |
| 1b | Apply the phone/laptop title rules, price floor and brand info to the prefiltered lines (local, fast) | `data/raw/meta_candidates.parquet` |
| 2 | Stream each review file, keep only reviews of those candidates (resumable) | `data/raw/reviews_candidates_*.jsonl` |
| 3 | Count reviews, history and supported quarters per `parent_asin`, apply the rules, pick the subset | in memory |
| 4 | Save the selection and its reviews, draw the summary figure | `data/processed/products_selected.csv`, `data/raw/reviews_selected.parquet`, `outputs/figures/1_selected_products.png` |

The review files are large (Electronics is tens of GB). Steps 1a and 2 stream them once and keep only what is
needed. **They are resumable**: progress is checkpointed next to the output, so if you stop the cell (or the
connection drops) the next run continues from the last checkpoint instead of starting again. Steps whose output
already exists are skipped.

**Slow home connection?** Run steps 1a–2 on Colab or Kaggle (much faster download), then copy
`meta_candidates.parquet` (or the two `meta_*_prefilter.jsonl` files) and the two `reviews_candidates_*.jsonl`
files into your local `data/raw/` and run the rest here. Optionally set the `HF_TOKEN` environment variable to
avoid anonymous download throttling.

## Configuration

In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# Project paths (works whether the notebook is started from notebooks/ or the project root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
for folder in (RAW, PROCESSED, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

# Amazon Reviews 2023 raw files. A file already present in data/raw (same file name) is used
# instead of downloading, so you can also fetch the files by hand.
HF_RAW = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw"
HEADERS = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"} if os.environ.get("HF_TOKEN") else {}

# dataset category -> product type we take from it
CATEGORIES = {
    "Cell_Phones_and_Accessories": "smartphone",
    "Electronics": "laptop",
}

# Selection rules
MIN_REVIEWS = 500            # reviews per parent_asin
MIN_HISTORY_YEARS = 3.0      # first to last review
SUPPORTED_QUARTER_MIN = 10   # a quarter "has support" with at least this many reviews
MIN_SUPPORTED_QUARTERS = 10  # supported quarters needed (rolling-origin cutoffs)
MIN_PRICE = {"smartphone": 100.0, "laptop": 200.0}   # USD; unknown price is kept and flagged
TARGET_PER_TYPE = 25         # 2 types x 25 = 50 (upper end of the 30-50 target)
MAX_PER_BRAND = 5            # keeps the subset from being one brand
PP1_PER_TYPE = 5             # development subset: 4 high-support + 1 sparse showcase per type
RESTRICT_TO_SCANNED = True   # reuse a cached review scan even if the candidate list changed (drops products it lacks)
PP1_MAX_PER_BRAND = 2
CANDIDATE_POOL = 150         # per type, top by rating_number; bounds the review scan
SPARSE_QUARTER_THRESHOLD = 5 # a quarter with fewer reviews than this counts as sparse
EXCLUDE_REFURBISHED = True   # renewed/refurbished listings mix seller condition into the reviews
# parent_asin values to drop after eyeballing the result. The title rules let these through ("flex" is a
# laptop series keyword; GSM / prepaid words pass the phone evidence), so they are removed by hand.
MANUAL_EXCLUDE = {
    "B07MLY3JKV",  # Echo Flex smart speaker
    "B00BGO0QEO",  # Fitbit Flex activity tracker
    "B07K9DVZ3H",  # Canary Flex security camera
    "B0067XXRZO",  # Joby Gorillapod tripod
    "B081JVRJZR",  # RCA to HDMI converter
    "B07QN8HY56",  # Sennheiser Momentum 2.0 headphones
    "B08BNB4KY3",  # Sennheiser Momentum 3 headphones
    "B013OUS19E",  # h2o SIM starter kit
    "B0C7ZH5TGD",  # Tracfone SIM kit
}
CHECKPOINT_BYTES = 256 << 20 # save scan progress every 256 MiB read

## Helpers

* `iter_lines` streams a `.jsonl` file from disk or over HTTP and reports the exact byte length of every line
  (with retry/resume if the connection drops).
* `stream_filter` copies the wanted lines of a huge file into a small file and **checkpoints** its position, so a
  stopped run resumes where it left off.
* `is_target_product` separates real phones/laptops from the accessories that share the same Amazon categories.
  The `categories` field is often empty in the 2023 metadata, so the product title does most of the work. The
  selected titles are printed at the end so this can be checked by eye.
* `make_review_id` gives every review a stable ID (the dataset has none). `user_id` is never stored.

In [ ]:
def source_for(kind, category):
    """Local file in data/raw if present, otherwise the Hugging Face URL."""
    if kind == "meta":
        name, url = f"meta_{category}.jsonl", f"{HF_RAW}/meta_categories/meta_{category}.jsonl"
    else:
        name, url = f"{category}.jsonl", f"{HF_RAW}/review_categories/{category}.jsonl"
    local = RAW / name
    return local if local.exists() else url


def iter_lines(source, start=0, max_retries=10):
    """Yield (line, nbytes) of a local or remote .jsonl file, beginning at byte offset `start`.

    nbytes = raw bytes the line occupied including its newline, so callers can keep an exact position.
    Empty lines are yielded too (callers skip them) so the byte count never drifts.
    """
    if isinstance(source, Path):
        size = source.stat().st_size
        with open(source, "rb") as f, tqdm(total=size, initial=start, unit="B", unit_scale=True,
                                           desc=source.name) as bar:
            f.seek(start)
            pending = 0
            for line in f:
                pending += len(line)
                if pending >= (1 << 22):
                    bar.update(pending)
                    pending = 0
                yield line.rstrip(b"\r\n"), len(line)
            bar.update(pending)
        return

    offset, remainder, retries, total = start, b"", 0, None
    bar = tqdm(unit="B", unit_scale=True, desc=source.rsplit("/", 1)[-1], initial=start)
    try:
        while True:
            try:
                headers = dict(HEADERS)
                if offset:
                    headers["Range"] = f"bytes={offset}-"
                with requests.get(source, headers=headers, stream=True, timeout=60) as resp:
                    resp.raise_for_status()
                    if offset and resp.status_code != 206:
                        raise RuntimeError("The server ignored the Range header, so the download cannot resume.")
                    if total is None:
                        total = int(resp.headers.get("Content-Length", 0)) + offset
                        bar.total = total or None
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        offset += len(chunk)
                        bar.update(len(chunk))
                        *lines, remainder = (remainder + chunk).split(b"\n")
                        for line in lines:
                            yield line.rstrip(b"\r"), len(line) + 1
                if remainder:
                    yield remainder.rstrip(b"\r"), len(remainder)
                return
            except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError):
                retries += 1
                if retries > max_retries:
                    raise
                time.sleep(min(30, 2 ** retries))  # then resume from `offset`
    finally:
        bar.close()


def stream_filter(source, out_path, keep, tag):
    """Copy the lines of a (huge) .jsonl for which keep(line) is true into `out_path`.

    Resumable: every CHECKPOINT_BYTES the position is saved next to the output (`.ckpt.json`). If the run is
    stopped, the next call continues from the last checkpoint (the output is cut back to that point first, so no
    line is duplicated). `tag` identifies what is being filtered; a checkpoint with a different tag is ignored.
    """
    out_path = Path(out_path)
    part, ckpt = out_path.with_suffix(".part"), out_path.with_suffix(".ckpt.json")
    state = {"tag": tag, "offset": 0, "part_bytes": 0, "kept": 0}
    if ckpt.exists() and part.exists():
        saved = json.loads(ckpt.read_text())
        if saved.get("tag") == tag:
            state = saved
            with open(part, "r+b") as f:
                f.truncate(state["part_bytes"])
            print(f"{out_path.name}: resuming from byte {state['offset']:,} ({state['kept']:,} lines kept so far)")
    pos, since = state["offset"], 0
    with open(part, "ab" if pos else "wb") as out:
        for line, n in iter_lines(source, start=pos):
            if line and keep(line):
                out.write(line + b"\n")
                state["kept"] += 1
            pos += n
            since += n
            if since >= CHECKPOINT_BYTES:
                out.flush()
                state.update(offset=pos, part_bytes=out.tell())
                ckpt.write_text(json.dumps(state))
                since = 0
    part.replace(out_path)
    ckpt.unlink(missing_ok=True)
    print(f"{out_path.name}: done, kept {state['kept']:,} lines")
    return state["kept"]


def to_price(value):
    if value is None:
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        m = re.search(r"\d[\d,]*\.?\d*", str(value))
        return float(m.group().replace(",", "")) if m else np.nan


def make_review_id(user_id, asin, timestamp):
    """Stable review ID (the dataset has none); does not depend on file order. user_id is not stored."""
    return hashlib.sha1(f"{user_id}|{asin}|{timestamp}".encode("utf-8")).hexdigest()[:16]


# Words that mark an accessory wherever they appear in the title.
STRONG_ACCESSORY = (r"cases?|covers?|protectors?|tempered|chargers?|cables?|adapters?|holders?|mounts?|stands?|"
                    r"sleeves?|bags?|backpacks?|stickers?|decals?|skins?|replacements?|repair|boosters?|antennas?|"
                    r"repeaters?|power bank|docks?|docking|lanyard|armband|screen guard|cooling pad")
# Words that mark a different product only when they are in the product name (the part before the first
# comma / dash / bracket). Real devices often mention "wireless charging" or "backlit keyboard" later in the title.
HEAD_ACCESSORY = {
    "smartphone": r"charging|earbuds?|buds|headphones?|earphones?|headsets?|stylus|wallet|pouch|bumper|camera lens|"
                  r"lens kit|car kit|film|sim (?:card )?tray|ejector|selfie|tripod|gimbal|ring light|cleaning|cleaner|"
                  r"watch|smartwatch|tablet|tab|ipad|speakers?|speakerphone|fit|tracker|keyboard|mouse|strap|cradle|"
                  r"grip|battery|memory|micro ?sd|sd card|flash drive",
    "laptop": r"mouse|mice|keyboard|headphones?|earphones?|earbuds?|headsets?|memory|ram|ssd|hdd|hard drive|drive|"
              r"projector|speakers?|webcam|monitor|printer|tablet|ipad|desktop|dongle|enclosure|desk|cart|stylus|"
              r"cleaner|cleaning|battery|virus|usb",
}
# A product must also look like the real device. Phones: phone words, "unlocked", or a storage size.
# Laptops: a laptop word or a laptop series name in the product name.
PHONE_EVIDENCE = (r"smartphone|smart phone|cell ?phone|mobile phone|android phone|unlocked|prepaid|tracfone|"
                  r"straight talk|gsm|dual sim|\b\d{2,4} ?gb\b")
LAPTOP_HEAD_EVIDENCE = (r"laptop|notebook|chromebook|macbook|ultrabook|pixelbook|aspire|predator|nitro|swift|"
                        r"thinkpad|thinkbook|ideapad|yoga|legion|flex|inspiron|xps|latitude|pavilion|envy|spectre|"
                        r"omen|victus|zenbook|vivobook|surface|galaxy book|elitebook|probook|travelmate|extensa|"
                        r"alienware|razer blade|\bgram\b")
INCLUDE_CATEGORY = {
    "smartphone": {"cell phones", "unlocked cell phones", "smartphones"},
    "laptop": {"laptops", "traditional laptops", "2 in 1 laptops", "chromebooks"},
}
# "for PC Laptop", "for Phones", "compatible with iPhone", "for Android TV": the product is made for the device.
# At most one word may sit between "for" and the device ("for US by Motorola" is a real phone title and passes).
FOR_DEVICE = re.compile(
    r"\b(?:for|compatible with|fits?)\s+(?:the\s+)?(?:\w+\s+)?"
    r"(?:iphones?|ipads?|samsung|galaxy|google|pixel|moto|motorola|oneplus|nokia|huawei|phones?|cell|smartphones?|"
    r"android tv|tv|pcs?|macs?|macbooks?|laptops?|notebooks?|chromebooks?|computers?|desktops?|tablets?)\b", re.I)
REFURBISHED = re.compile(r"renewed|refurbished|pre-?owned|open box|reconditioned", re.I)


def title_head(title):
    """Product name: the title up to the first comma, bracket, pipe, spaced dash, "with" or a storage size."""
    return re.split(r"\s[-–—]\s|[,|(\[]|:\s|\swith\s|\s\d+\s?(?:gb|tb)\b", title, maxsplit=1, flags=re.I)[0]


def is_target_product(title, categories, product_type):
    t = title.lower()
    head = title_head(title).lower()
    if re.search(rf"\b(?:{STRONG_ACCESSORY})\b", t):
        return False
    if re.search(rf"\b(?:{HEAD_ACCESSORY[product_type]})\b", head):
        return False
    if FOR_DEVICE.search(title):
        return False
    cats = {c.strip().lower() for c in categories}
    if cats & INCLUDE_CATEGORY[product_type]:
        return True
    if product_type == "smartphone":
        return bool(re.search(PHONE_EVIDENCE, t))
    return bool(re.search(LAPTOP_HEAD_EVIDENCE, head))

## Step 1 · Metadata → candidate products

**1a.** Stream each metadata file and keep only the lines with at least `MIN_REVIEWS` ratings
(`rating_number` is an upper bound for the number of reviews, so this is a safe first filter). This is the only
step that downloads the metadata; it is resumable and cached.

**1b.** Apply the phone/laptop title rules, the price floor and the refurbished rule to those lines. This works
on a small local file, so you can change the keyword lists, `MIN_PRICE` or `CANDIDATE_POOL` and just delete
`meta_candidates.parquet` to redo it in seconds, without downloading anything again. Per type, only the
`CANDIDATE_POOL` products with the most ratings go on to the review scan.

In [ ]:
RATING_RE = re.compile(rb'"rating_number": ?(\d+)')


def keep_if_enough_ratings(line):
    m = RATING_RE.search(line)                  # cheap pre-check before parsing the whole record
    return bool(m) and int(m.group(1)) >= MIN_REVIEWS


def prefilter_meta(category):
    out = RAW / f"meta_{category}_prefilter.jsonl"
    side = out.with_suffix(".json")
    if out.exists():
        saved = json.loads(side.read_text())["min_reviews"] if side.exists() else None
        if saved is not None and saved > MIN_REVIEWS:
            raise RuntimeError(f"{out.name} was filtered with MIN_REVIEWS={saved}. Delete it (and {side.name}) "
                               f"to rescan with MIN_REVIEWS={MIN_REVIEWS}.")
        return out
    stream_filter(source_for("meta", category), out, keep_if_enough_ratings, tag=f"{category}|{MIN_REVIEWS}")
    side.write_text(json.dumps({"min_reviews": MIN_REVIEWS}))
    return out


def candidates_from_prefilter(path, product_type):
    rows = []
    with open(path, "rb") as f:
        for line in f:
            rec = json.loads(line)
            if int(rec.get("rating_number") or 0) < MIN_REVIEWS:
                continue
            title = rec.get("title") or ""
            categories = rec.get("categories") or []
            if not title or not is_target_product(title, categories, product_type):
                continue
            refurbished = bool(REFURBISHED.search(title))
            if EXCLUDE_REFURBISHED and refurbished:
                continue
            price = to_price(rec.get("price"))
            if not np.isnan(price) and price < MIN_PRICE[product_type]:
                continue                            # accessories are cheap; unknown price stays and is flagged
            rows.append({
                "parent_asin": rec["parent_asin"],
                "product_type": product_type,
                "title": title,
                "store": (rec.get("store") or "").strip(),
                "price": price,
                "price_known": not np.isnan(price),
                "average_rating": rec.get("average_rating"),
                "rating_number": rec.get("rating_number"),
                "is_refurbished": refurbished,
                "category_path": " > ".join(categories),
            })
    return pd.DataFrame(rows)


META_CACHE = RAW / "meta_candidates.parquet"
if META_CACHE.exists():
    meta = pd.read_parquet(META_CACHE)
else:
    parts = []
    for category, product_type in CATEGORIES.items():
        prefiltered = prefilter_meta(category)
        df = candidates_from_prefilter(prefiltered, product_type)
        df = df.drop_duplicates("parent_asin").sort_values("rating_number", ascending=False).head(CANDIDATE_POOL)
        parts.append(df)
    meta = pd.concat(parts, ignore_index=True)
    meta.to_parquet(META_CACHE, index=False)

print(meta.groupby("product_type").size().rename("candidates").to_string())
meta[["product_type", "title", "store", "price", "rating_number"]].head(15)

meta_Cell_Phones_and_Accessories.jsonl: 0.00B [00:00, ?B/s]

KeyboardInterrupt: 

## Step 2 · Reviews of the candidates

Stream each review file and copy only the lines whose `parent_asin` is a candidate. Lines are matched with a
regular expression instead of being parsed as JSON, which keeps the scan of the very large files cheap. The scan
is resumable (checkpoint every 256 MiB), so stopping the cell is safe: run it again and it continues.

The candidate list is remembered next to each output. If you later change the candidate rules so that products
appear that were not scanned, the notebook tells you to delete the review file and rescan; a smaller candidate
list reuses the cached scan.

In [ ]:
PARENT_RE = re.compile(rb'"parent_asin": ?"([A-Za-z0-9]+)"')

review_files = []
for category, product_type in CATEGORIES.items():
    out_path = RAW / f"reviews_candidates_{product_type}.jsonl"
    side = out_path.with_suffix(".wanted.json")
    review_files.append(out_path)
    wanted = set(meta.loc[meta["product_type"] == product_type, "parent_asin"])

    if out_path.exists():
        scanned = set(json.loads(side.read_text())) if side.exists() else None
        if scanned is not None and not wanted <= scanned:
            missing = wanted - scanned
            if not RESTRICT_TO_SCANNED:
                raise RuntimeError(f"{out_path.name} was scanned for a different candidate list "
                                   f"({len(missing)} new products). Delete it (and {side.name}) and rerun this cell.")
            meta = meta[~meta["parent_asin"].isin(missing)]          # no reviews were saved for them
            print(f"{out_path.name}: cached. {len(missing)} candidate(s) are not in this scan and are skipped "
                  f"(RESTRICT_TO_SCANNED=True). Set it to False and delete the file to scan them.")
        else:
            print(f"{out_path.name}: cached")
        continue

    wanted_b = {w.encode() for w in wanted}

    def keep(line, wanted_b=wanted_b):
        m = PARENT_RE.search(line)
        return bool(m) and m.group(1) in wanted_b

    tag = f"{category}|{hashlib.sha1(','.join(sorted(wanted)).encode()).hexdigest()[:12]}"
    kept = stream_filter(source_for("review", category), out_path, keep, tag)
    side.write_text(json.dumps(sorted(wanted)))
    print(f"{out_path.name}: kept {kept:,} reviews of {len(wanted)} candidate products")

## Step 3 · Per-product statistics and selection

Per `parent_asin`: number of reviews, number of colour/size variants (`asin`) and the share of the biggest variant,
first and last review, history length, and how many quarters in that span are sparse or supported. Review
timestamps are in milliseconds. Raw counts are used here; empty and duplicate reviews are removed in the cleaning
step.

In [ ]:
def load_review_index(paths):
    parent, asin, ts = [], [], []
    for path in paths:
        with open(path, "rb") as f:
            for line in f:
                r = json.loads(line)
                parent.append(r["parent_asin"])
                asin.append(r["asin"])
                ts.append(r["timestamp"])
    df = pd.DataFrame({"parent_asin": parent, "asin": asin, "timestamp": ts})
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["quarter"] = df["date"].dt.to_period("Q")
    return df


def product_stats(index):
    rows = []
    for parent_asin, g in index.groupby("parent_asin"):
        per_quarter = g.groupby("quarter").size()
        span = pd.period_range(per_quarter.index.min(), per_quarter.index.max(), freq="Q")
        per_quarter = per_quarter.reindex(span, fill_value=0)   # quarters with no reviews count too
        variants = g["asin"].value_counts()
        rows.append({
            "parent_asin": parent_asin,
            "n_reviews": len(g),
            "n_variants": len(variants),
            "top_variant_share": round(float(variants.iloc[0] / len(g)), 3),
            "first_review": g["date"].min().date(),
            "last_review": g["date"].max().date(),
            "history_years": round((g["date"].max() - g["date"].min()).days / 365.25, 2),
            "n_quarters_span": len(span),
            "n_active_quarters": int((per_quarter > 0).sum()),
            "n_supported_quarters": int((per_quarter >= SUPPORTED_QUARTER_MIN).sum()),
            "sparse_quarter_share": round(float((per_quarter < SPARSE_QUARTER_THRESHOLD).mean()), 3),
        })
    return pd.DataFrame(rows)


review_index = load_review_index(review_files)
stats = product_stats(review_index)
candidates = meta.merge(stats, on="parent_asin", how="inner")
print(f"{len(review_index):,} reviews over {len(candidates)} candidate products")

candidates["eligible"] = (
    (candidates["n_reviews"] >= MIN_REVIEWS)
    & (candidates["history_years"] >= MIN_HISTORY_YEARS)
    & (candidates["n_supported_quarters"] >= MIN_SUPPORTED_QUARTERS)
    & ~candidates["parent_asin"].isin(MANUAL_EXCLUDE)
)
print(candidates.groupby("product_type")["eligible"].agg(candidates="size", eligible="sum").to_string())

In [ ]:
def brand_of(g):
    return g["store"].replace("", "unknown").str.lower()


selected_parts = []
for product_type, g in candidates[candidates["eligible"]].groupby("product_type"):
    g = g.sort_values("n_reviews", ascending=False)
    g = g[g.groupby(brand_of(g)).cumcount() < MAX_PER_BRAND]       # brand cap, keeps the best-reviewed ones
    selected_parts.append(g.head(TARGET_PER_TYPE))

selected = pd.concat(selected_parts, ignore_index=True)


def pick_pp1(g):
    """PP1 development subset: high-support products (brand cap) plus one sparse showcase."""
    g = g.sort_values("n_reviews", ascending=False)
    dense = g[g.groupby(brand_of(g)).cumcount() < PP1_MAX_PER_BRAND].head(PP1_PER_TYPE - 1)
    rest = g[~g["parent_asin"].isin(dense["parent_asin"])]
    sparse = rest.sort_values("sparse_quarter_share", ascending=False).head(1)
    return pd.concat([dense.assign(pp1_role="high_support"), sparse.assign(pp1_role="sparse_showcase")])


pp1 = pd.concat([pick_pp1(g) for _, g in selected.groupby("product_type")])
selected = selected.merge(pp1[["parent_asin", "pp1_role"]], on="parent_asin", how="left")
selected["pp1_subset"] = selected["pp1_role"].notna()

columns = [
    "parent_asin", "product_type", "pp1_subset", "pp1_role", "title", "store", "price", "price_known",
    "average_rating", "rating_number", "n_reviews", "n_variants", "top_variant_share",
    "first_review", "last_review", "history_years", "n_quarters_span", "n_active_quarters",
    "n_supported_quarters", "sparse_quarter_share",
]
selected = selected[columns].sort_values(["product_type", "n_reviews"], ascending=[True, False]).reset_index(drop=True)

print(f"Selected {len(selected)} products ({int(selected['pp1_subset'].sum())} in the PP1 development subset)")
if not 30 <= len(selected) <= 50:
    print("WARNING: outside the 30-50 target. Relax MIN_REVIEWS / MIN_HISTORY_YEARS / MAX_PER_BRAND "
          "or raise CANDIDATE_POOL, then delete data/raw/meta_candidates.parquet and rerun Step 1 (no new download "
          "unless the pool grows beyond the scanned reviews, in which case Step 2 tells you).")
selected[["product_type", "pp1_subset", "title", "store", "price", "n_reviews", "history_years",
          "n_supported_quarters", "sparse_quarter_share"]]

### Sanity checks

Read the titles above once: if an accessory slipped through, add its `parent_asin` to `MANUAL_EXCLUDE`
(or extend the keyword lists) and re-run from Step 1b. Also look at the variant columns: many variants with a low
`top_variant_share` can mean one listing mixes different models, which would look like a fake "change" in the
time series. Note anything suspicious in the decision log.

In [ ]:
assert selected["parent_asin"].is_unique
assert (selected["n_reviews"] >= MIN_REVIEWS).all()
assert (selected["history_years"] >= MIN_HISTORY_YEARS).all()
assert (selected["n_supported_quarters"] >= MIN_SUPPORTED_QUARTERS).all()
assert selected.groupby("product_type").size().le(TARGET_PER_TYPE).all()

summary = selected.groupby("product_type").agg(
    products=("parent_asin", "size"),
    reviews=("n_reviews", "sum"),
    median_reviews=("n_reviews", "median"),
    median_history_years=("history_years", "median"),
    median_sparse_share=("sparse_quarter_share", "median"),
    pp1_products=("pp1_subset", "sum"),
)
display(summary)

flag = selected[(selected["n_variants"] > 10) & (selected["top_variant_share"] < 0.5)]
print("Variant check:", "no product with many evenly spread variants" if flag.empty else
      f"{len(flag)} product(s) with >10 variants and no dominant one - inspect:")
if not flag.empty:
    display(flag[["parent_asin", "title", "n_variants", "top_variant_share"]])
print("Products with unknown price (check the title by eye):", int((~selected["price_known"]).sum()))

## Step 4 · Save the selection

* `data/processed/products_selected.csv` – the selected products and their statistics (`pp1_subset` marks the
  5 + 5 development subset).
* `data/raw/reviews_selected.parquet` – all reviews of the selected products, input for the cleaning notebook.
  Each review gets a stable `review_id` (sha1 of user_id | asin | timestamp, first 16 characters); `user_id`
  itself is **not** stored. Share this ID format with the Component 1 owner.
* `data/processed/selection_log.json` – the rules and parameters used (reproducibility).

In [ ]:
selected.to_csv(PROCESSED / "products_selected.csv", index=False)

keep = set(selected["parent_asin"])
type_of = dict(zip(selected["parent_asin"], selected["product_type"]))
fields = ["parent_asin", "asin", "rating", "title", "text", "timestamp", "helpful_vote", "verified_purchase"]
rows = []
for path in review_files:
    with open(path, "rb") as f:
        for line in f:
            r = json.loads(line)
            if r["parent_asin"] in keep:
                row = {k: r.get(k) for k in fields}
                row["review_id"] = make_review_id(r.get("user_id"), r.get("asin"), r.get("timestamp"))
                rows.append(row)

reviews_selected = pd.DataFrame(rows).rename(columns={"title": "review_title"})
n_raw = len(reviews_selected)
reviews_selected = reviews_selected.drop_duplicates("review_id").reset_index(drop=True)
reviews_selected.insert(0, "review_id", reviews_selected.pop("review_id"))
reviews_selected.insert(1, "product_type", reviews_selected["parent_asin"].map(type_of))
reviews_selected["quarter"] = pd.to_datetime(reviews_selected["timestamp"], unit="ms").dt.to_period("Q").astype(str)
reviews_selected.to_parquet(RAW / "reviews_selected.parquet", index=False)

assert "user_id" not in reviews_selected.columns
assert n_raw == selected["n_reviews"].sum()
print(f"products_selected.csv: {len(selected)} products")
print(f"reviews_selected.parquet: {len(reviews_selected):,} reviews ({n_raw - len(reviews_selected):,} duplicate review_ids removed)")

log = {
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "dataset": "McAuley-Lab/Amazon-Reviews-2023",
    "parameters": {
        "MIN_REVIEWS": MIN_REVIEWS, "MIN_HISTORY_YEARS": MIN_HISTORY_YEARS,
        "SUPPORTED_QUARTER_MIN": SUPPORTED_QUARTER_MIN, "MIN_SUPPORTED_QUARTERS": MIN_SUPPORTED_QUARTERS,
        "MIN_PRICE": MIN_PRICE, "TARGET_PER_TYPE": TARGET_PER_TYPE, "MAX_PER_BRAND": MAX_PER_BRAND,
        "PP1_PER_TYPE": PP1_PER_TYPE, "PP1_MAX_PER_BRAND": PP1_MAX_PER_BRAND,
        "CANDIDATE_POOL": CANDIDATE_POOL, "EXCLUDE_REFURBISHED": EXCLUDE_REFURBISHED,
    },
    "manual_exclude": sorted(MANUAL_EXCLUDE),
    "n_candidates": int(len(candidates)),
    "n_eligible": int(candidates["eligible"].sum()),
    "n_selected": int(len(selected)),
    "n_reviews": int(len(reviews_selected)),
    "pp1_subset": selected.loc[selected["pp1_subset"], ["parent_asin", "product_type", "pp1_role", "title"]].to_dict("records"),
}
(PROCESSED / "selection_log.json").write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(6, 0.22 * len(selected))),
                               gridspec_kw={"width_ratios": [3, 2]})
colors = {"smartphone": "#1f77b4", "laptop": "#ff7f0e"}

ordered = selected.sort_values("n_reviews").reset_index(drop=True)
ypos = np.arange(len(ordered))            # positions, not titles: equal titles would otherwise merge into one bar
ax1.barh(ypos, ordered["n_reviews"], color=ordered["product_type"].map(colors),
         edgecolor=np.where(ordered["pp1_subset"], "black", "none"), linewidth=1.2)
ax1.set_yticks(ypos)
ax1.set_yticklabels(ordered["title"].str.slice(0, 45), fontsize=7)
ax1.set_xlabel("Reviews")
ax1.set_title("Reviews per selected product (black outline = PP1 development subset)")

for product_type, g in selected.groupby("product_type"):
    ax2.scatter(g["history_years"], g["sparse_quarter_share"], label=product_type,
                color=colors[product_type], alpha=0.8)
ax2.set_xlabel("Review history (years)")
ax2.set_ylabel(f"Share of quarters with < {SPARSE_QUARTER_THRESHOLD} reviews")
ax2.set_title("History length vs sparsity")
ax2.legend()

fig.tight_layout()
fig.savefig(FIGURES / "1_selected_products.png", dpi=150)
plt.show()